# IMU Analysis - MWDPlanet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.size':        18,
    'axes.titlesize':   22,
    'axes.labelsize':   18,
    'xtick.labelsize':  16,
    'ytick.labelsize':  16,
    'legend.fontsize':  16,
    'figure.titlesize': 26,
    'lines.linewidth':  2.0,
})

df = pd.read_csv('imu_challenge_dataset_ext.csv', skiprows=1)

t = df['time_s'].values
data_cols = [c for c in df.columns if c != 'time_s']

print(f'Samples : {len(df):,}')
print(f'Duration: {t[-1]:.1f} s')
print(f'Columns : {data_cols}')
df.head(3)

---
## Section 1: Raw Signal

In [ ]:
n = len(data_cols)
fig, axes = plt.subplots(n, 1, figsize=(14, 3 * n), sharex=True)

for ax, col in zip(axes, data_cols):
    ax.plot(t, df[col].values, linewidth=0.7)
    ax.set_title(col, fontweight='bold')
    ax.set_ylabel(col)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)')
fig.suptitle('IMU Raw', y=1.01)
plt.tight_layout()
plt.savefig('all_channels.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 2: Gyro-Only

$$\theta(t) = \int_0^t \omega(\tau)\, d\tau \approx \sum_k \omega_k \, \Delta t_k$$

> **Limitation:** gyro integration accumulates drift over time (no correction signal).

In [ ]:
dt = np.diff(t, prepend=t[0])

roll_gyro  = np.degrees(np.cumsum(df['gx_rad_s'].values * dt))
pitch_gyro = np.degrees(np.cumsum(df['gy_rad_s'].values * dt))
yaw_gyro   = np.degrees(np.cumsum(df['gz_rad_s'].values * dt))

gyro_angles = [
    ('Roll  (gx)', roll_gyro,  'tab:blue'),
    ('Pitch (gy)', pitch_gyro, 'tab:orange'),
    ('Yaw   (gz)', yaw_gyro,   'tab:green'),
]

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
for ax, (label, data, color) in zip(axes, gyro_angles):
    ax.plot(t, data, color=color)
    ax.set_title(label, fontweight='bold')
    ax.set_ylabel('Angle (deg)')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Gyro-Only Attitude (Dead-Reckoning)', y=1.01)
plt.tight_layout()
plt.savefig('gyro_attitude.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 3: Accelerometer-Only

$$\text{Roll}  = \mathrm{atan2}(a_y,\, a_z)$$
$$\text{Pitch} = \mathrm{atan2}\!\left(-a_x,\, \sqrt{a_y^2 + a_z^2}\right)$$
$$\text{Yaw} = \mathrm{atan2}(a_x,\, a_y)$$

> **Yaw caveat:** This is not a good approximation for Yaw.  
> **limitation:** any linear acceleration contaminates all estimates (not drift-free like gyro, but also not divergent).

In [ ]:
ax_  = df['ax_m_s2'].values
ay_  = df['ay_m_s2'].values
az_  = df['az_m_s2'].values

roll_acc  = np.degrees(np.arctan2(ay_, az_))
pitch_acc = np.degrees(np.arctan2(-ax_, np.sqrt(ay_**2 + az_**2)))
yaw_acc   = np.degrees(np.arctan2(ax_, ay_))

acc_angles = [
    ('Roll  (atan2(ay, az))', roll_acc, 'tab:blue'),
    ('Pitch (atan2(-ax, sqrt(ay^2+az^2)))', pitch_acc, 'tab:orange'),
    ('Yaw   (atan2(ax, ay))', yaw_acc, 'tab:green'),
]

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
for ax, (label, data, color) in zip(axes, acc_angles):
    ax.plot(t, data, color=color)
    ax.set_title(label, fontweight='bold')
    ax.set_ylabel('Angle (deg)')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Accelerometer-Only Attitude (atan2)', y=1.01)
plt.tight_layout()
plt.savefig('acc_attitude.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 4: Complementary Filter (Gyro + Accelerometer)

$$\theta_k = \alpha \,(\theta_{k-1} + \omega_k \,\Delta t) + (1-\alpha)\,\theta_{\text{acc},k}$$

> \alpha balances the contributions of the gyroscope and accelerometer. The gyroscope is accurate in the short term but drifts over time, while the accelerometer provides an absolute orientation reference but is noisy.

In [ ]:
ALPHA = 0.98

gx = df['gx_rad_s'].values
gy = df['gy_rad_s'].values
gz = df['gz_rad_s'].values

n_samples = len(t)
roll_cf  = np.zeros(n_samples)
pitch_cf = np.zeros(n_samples)
yaw_cf   = np.zeros(n_samples)

roll_cf[0]  = roll_acc[0]
pitch_cf[0] = pitch_acc[0]
yaw_cf[0]   = yaw_acc[0]

for i in range(1, n_samples):
    d = dt[i]
    roll_cf[i]  = ALPHA * (roll_cf[i-1]  + np.degrees(gx[i] * d)) + (1 - ALPHA) * roll_acc[i]
    pitch_cf[i] = ALPHA * (pitch_cf[i-1] + np.degrees(gy[i] * d)) + (1 - ALPHA) * pitch_acc[i]
    yaw_cf[i]   = ALPHA * (yaw_cf[i-1]   + np.degrees(gz[i] * d)) + (1 - ALPHA) * yaw_acc[i]

cf_angles = [
    ('Roll',  roll_cf,  roll_gyro,  roll_acc,  'tab:blue'),
    ('Pitch', pitch_cf, pitch_gyro, pitch_acc, 'tab:orange'),
    ('Yaw',   yaw_cf,   yaw_gyro,   yaw_acc,   'tab:green'),
]

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
for ax, (label, cf, gyro, acc, color) in zip(axes, cf_angles):
    ax.plot(t, gyro, linewidth=1.2, color='grey',  alpha=0.5, label='Gyro only')
    ax.plot(t, acc,  linewidth=1.2, color='salmon', alpha=0.5, label='Acc only')
    ax.plot(t, cf,   linewidth=2.5, color=color,   label=f'Complementary (α={ALPHA})')
    ax.set_title(label, fontweight='bold')
    ax.set_ylabel('Angle (deg)')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)')
fig.suptitle(f'Complementary Filter Attitude  (α = {ALPHA})', y=1.01)
plt.tight_layout()
plt.savefig('complementary_filter.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 5: Kalman Filter

The first 20s the sensor is stationary, during which:
- We measured gyro data variance for Q
- and measured accelaration data variance for R


$$\mathbf{x} = \begin{bmatrix}\theta\\b\end{bmatrix}$$

$$\mathbf{x}_k^- = F\,\mathbf{x}_{k-1} + \mathbf{B}\,\omega_k, \quad
F = \begin{bmatrix}1 & -\Delta t\\0 & 1\end{bmatrix}, \quad
\mathbf{B} = \begin{bmatrix}\Delta t\\0\end{bmatrix}$$

$$P^-_k = FP_kF^T + Q$$

$$K_k = P_k^- H^T (H P_k^- H^T + R)^{-1}$$
$$\mathbf{x}_k = \mathbf{x}_k^- + K_k\,(H\,\mathbf{x}_k - H\,\mathbf{x}_k^-)$$

In [50]:
STATIC_END = 20
static = df[t < STATIC_END]

s_gx = static['gx_rad_s'].values
s_gy = static['gy_rad_s'].values
s_gz = static['gz_rad_s'].values
s_ax = static['ax_m_s2'].values
s_ay = static['ay_m_s2'].values
s_az = static['az_m_s2'].values

s_roll_acc  = np.degrees(np.arctan2(s_ay, s_az))
s_pitch_acc = np.degrees(np.arctan2(-s_ax, np.sqrt(s_ay**2 + s_az**2)))
s_yaw_acc   = np.degrees(np.arctan2(s_ax, s_ay))

var_gx_dps = np.var(np.degrees(s_gx))
var_gy_dps = np.var(np.degrees(s_gy))
var_gz_dps = np.var(np.degrees(s_gz))

R_roll  = np.var(s_roll_acc)
R_pitch = np.var(s_pitch_acc)
R_yaw   = np.var(s_yaw_acc)

Q_BIAS = 1e-6

print("── Q (process noise, gyro) ──────────────────")
print(f"var gx : {var_gx_dps:.6f}  (deg/s)^2")
print(f"var gy : {var_gy_dps:.6f}  (deg/s)^2")
print(f"var gz : {var_gz_dps:.6f}  (deg/s)^2")
print(f"Q_bias : {Q_BIAS:.2e}  (deg/s)^2")
print()
print("── R (measurement noise, acc angle) ─────────")
print(f"R roll  : {R_roll:.6f}  deg^2")
print(f"R pitch : {R_pitch:.6f}  deg^2")
print(f"R yaw   : {R_yaw:.6f}  deg^2")

── Q (process noise, gyro) ──────────────────
var gx : 0.012902  (deg/s)^2
var gy : 0.013106  (deg/s)^2
var gz : 0.012160  (deg/s)^2
Q_bias : 1.00e-06  (deg/s)^2

── R (measurement noise, acc angle) ─────────
R roll  : 0.085356  deg^2
R pitch : 0.088355  deg^2
R yaw   : 10618.559729  deg^2


In [ ]:
H = np.array([[1, 0]])

def kalman_axis(gyro_dps, acc_angle_deg, dt_arr, var_gyro_dps, R_meas):
    n = len(gyro_dps)
    angle = np.zeros(n)
    bias  = np.zeros(n)

    angle[0] = acc_angle_deg[0]
    bias[0]  = 0
    P = np.eye(2)

    for i in range(1, n):
        dt = dt_arr[i]
        F = np.array([[1, -dt], [0, 1]])
        B = np.array([dt, 0])
        Q = np.diag([var_gyro_dps * dt, Q_BIAS * dt])

        x_prev = np.array([angle[i-1], bias[i-1]])
        x_pred = F @ x_prev + B * gyro_dps[i]
        P_pred = F @ P @ F.T + Q

        S = (H @ P_pred @ H.T).item() + R_meas
        K = (P_pred @ H.T) / S
        innov = acc_angle_deg[i] - (H @ x_pred).item()
        x = x_pred + K.flatten() * innov
        P = (np.eye(2) - K @ H) @ P_pred

        angle[i] = x[0]
        bias[i]  = x[1]

    return angle, bias

gx_dps = np.degrees(df['gx_rad_s'].values)
gy_dps = np.degrees(df['gy_rad_s'].values)
gz_dps = np.degrees(df['gz_rad_s'].values)

roll_kf,  bias_roll  = kalman_axis(gx_dps, roll_acc,  dt, var_gx_dps, R_roll)
pitch_kf, bias_pitch = kalman_axis(gy_dps, pitch_acc, dt, var_gy_dps, R_pitch)
yaw_kf,   bias_yaw   = kalman_axis(gz_dps, yaw_acc,   dt, var_gz_dps, R_yaw)

kf_angles = [
    ('Roll',  roll_kf,  roll_gyro,  roll_acc,  bias_roll,  'tab:blue'),
    ('Pitch', pitch_kf, pitch_gyro, pitch_acc, bias_pitch, 'tab:orange'),
    ('Yaw',   yaw_kf,   yaw_gyro,   yaw_acc,   bias_yaw,   'tab:green'),
]

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
for ax, (label, kf, gyro, acc, bias, color) in zip(axes, kf_angles):
    ax.plot(t, gyro, linewidth=1.2, color='grey',  alpha=0.4, label='Gyro only')
    ax.plot(t, acc,  linewidth=1.2, color='salmon', alpha=0.4, label='Acc only')
    ax.plot(t, kf,   linewidth=2.5, color=color,   label='Kalman filter')
    ax.axvline(STATIC_END, color='k', linestyle='--', linewidth=1.5, label='Motion start')
    ax.set_title(label, fontweight='bold')
    ax.set_ylabel('Angle (deg)')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Kalman Filter', y=1.01)
plt.tight_layout()
plt.savefig('kalman_attitude.png', dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(3, 1, figsize=(14, 6), sharex=True)
for ax, (label, _, _, _, bias, color) in zip(axes, kf_angles):
    ax.plot(t, bias, color=color)
    ax.axvline(STATIC_END, color='k', linestyle='--', linewidth=1.5)
    ax.set_title(f'{label} — estimated gyro bias', fontweight='bold')
    ax.set_ylabel('Bias (deg/s)')
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('Time (s)')
fig.suptitle('Kalman Filter — Estimated Gyro Bias per Axis', y=1.01)
plt.tight_layout()
plt.savefig('kalman_bias.png', dpi=150, bbox_inches='tight')
plt.show()